In [24]:
"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║               TOOL-RAG  /  TOOLFORMER-RAG  ENGINE                             ║
║               tool_rag_engine.py  — Core Pipeline                             ║
║                                                                                ║
║  Inspired by:                                                                  ║
║    • Toolformer (Schick et al., 2023) — LM learns to call APIs inline         ║
║    • FLARE (Jiang et al., 2023)       — Active retrieval during generation     ║
║    • ReAct  (Yao et al., 2023)        — Reasoning + Acting interleaved         ║
║    • ToolkenGPT (Hao et al., 2023)   — Tool tokens embedded in LM vocab       ║
║                                                                                ║
║  LLM         : Azure OpenAI (GPT-4 / GPT-35-turbo)                            ║
║  Embeddings  : HuggingFace sentence-transformers (local, zero API cost)        ║
║  Vector DB   : ChromaDB (persistent HNSW index)                               ║
║  Framework   : LangChain (LCEL chains + custom tool dispatch)                 ║
╚══════════════════════════════════════════════════════════════════════════════════╝

WHAT MAKES THIS "TOOL-RAG" (NOT STANDARD RAG)
══════════════════════════════════════════════
Standard RAG  : retrieve-once → generate (fixed pipeline, no agency)
Agentic RAG   : LLM decides IF to retrieve (one loop before generation)
Tool-RAG      : LLM calls tools INLINE during generation, interleaving
                reasoning ↔ tool_calls ↔ observations in a single pass.
                Tools are first-class citizens: Calculator, KnowledgeBase,
                DateResolver, FactChecker, Summarizer, Comparator.
                The model generates "[TOOL: name(args)]" markers mid-text,
                the engine intercepts, executes, substitutes the result,
                and the model CONTINUES generating with real grounded context.

TOOLFORMER-STYLE INLINE TOOL CALLS
════════════════════════════════════
Generation step produces tokens like:
  "The Transformer has [TOOL:KB(multi-head attention)] and was trained
   for [TOOL:CALC(12 * 64)] = 768 hidden dims."

Engine intercepts → executes → substitutes → model resumes:
  "The Transformer has [Multi-Head Attention allows joint
   attention across 8 subspaces...] and was trained for 768 hidden dims."

TOOLS AVAILABLE
═══════════════
  1. KB         — Knowledge Base retrieval (ChromaDB semantic search)
  2. CALC       — Safe arithmetic evaluator
  3. DATE       — Date arithmetic / formatting
  4. COMPARE    — Side-by-side comparison via retrieved context
  5. SUMMARIZE  — Targeted summarization of retrieved chunks
  6. FACTCHECK  — Verify a claim against the knowledge base
  7. DEFINE     — Retrieve and format a definition/concept
"""


'\n╔══════════════════════════════════════════════════════════════════════════════════╗\n║               TOOL-RAG  /  TOOLFORMER-RAG  ENGINE                             ║\n║               tool_rag_engine.py  — Core Pipeline                             ║\n║                                                                                ║\n║  Inspired by:                                                                  ║\n║    • Toolformer (Schick et al., 2023) — LM learns to call APIs inline         ║\n║    • FLARE (Jiang et al., 2023)       — Active retrieval during generation     ║\n║    • ReAct  (Yao et al., 2023)        — Reasoning + Acting interleaved         ║\n║    • ToolkenGPT (Hao et al., 2023)   — Tool tokens embedded in LM vocab       ║\n║                                                                                ║\n║  LLM         : Azure OpenAI (GPT-4 / GPT-35-turbo)                            ║\n║  Embeddings  : HuggingFace sentence-transformers (local, zero API cost)   

In [25]:

# ─────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────
from __future__ import annotations

import os
import re
import ast
import sys
import json
import math
import time
import hashlib
import textwrap
import warnings
import operator
import datetime
import traceback
from enum import Enum
from pathlib import Path
from typing import (
    Any, Callable, Dict, List, Optional,
    Tuple, Union, NamedTuple
)
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")


In [26]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [27]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_classic.agents import AgentExecutor


In [28]:
import langchain_community

In [29]:
from langgraph.checkpoint.memory import InMemorySaver


In [30]:
import chromadb
from chromadb.config import Settings


In [31]:

# ─────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
@dataclass
class Config:
    """Central configuration. All values overridable via environment variables."""

    # ── Ollama (local LLM) ─────────────────────────────────────────────
    OLLAMA_BASE_URL: str = field(default_factory=lambda: os.getenv(
        "OLLAMA_BASE_URL", "http://localhost:11434"))
    OLLAMA_MODEL: str    = field(default_factory=lambda: os.getenv(
        "OLLAMA_MODEL", "qwen2.5-coder:7b"))

    # ── HuggingFace Embedding ─────────────────────────────────────────
    EMBEDDING_MODEL: str           = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str          = field(
        default_factory=lambda: os.getenv("EMBEDDING_DEVICE", "cpu"))

    # ── ChromaDB ─────────────────────────────────────────────────────
    CHROMA_PERSIST_DIR: str        = "./chroma_tool_rag"
    CHROMA_COLLECTION: str         = "tool_rag_corpus"

    # ── Retrieval ────────────────────────────────────────────────────
    TOP_K: int                     = 4
    FETCH_K: int                   = 8
    CHUNK_SIZE: int                = 700
    CHUNK_OVERLAP: int             = 120
    MMR_LAMBDA: float              = 0.65

    # ── Tool-RAG Loop ────────────────────────────────────────────────
    MAX_TOOL_ITERATIONS: int       = 6    # max tool calls per generation
    MAX_GENERATION_TOKENS: int     = 1200
    TOOL_CALL_TIMEOUT_SEC: float   = 15.0

    # ── LLM ──────────────────────────────────────────────────────────
    LLM_TEMPERATURE: float         = 0.0
    LLM_MAX_TOKENS: int            = 1200
    LLM_STREAMING: bool            = False

    def is_azure_configured(self) -> bool:
        return (
            "YOUR_AZURE" not in self.OLLAMA_BASE_URL
            and "YOUR_AZURE" not in self.OLLAMA_BASE_URL
        )


In [32]:

# ─────────────────────────────────────────────────────────────────────
# TERMINAL LOGGER
# ─────────────────────────────────────────────────────────────────────
class C:  # ANSI colour codes
    RESET  = "\033[0m"
    BOLD   = "\033[1m"
    DIM    = "\033[2m"
    CYAN   = "\033[96m"
    GREEN  = "\033[92m"
    YELLOW = "\033[93m"
    RED    = "\033[91m"
    MAGENTA= "\033[95m"
    BLUE   = "\033[94m"
    WHITE  = "\033[97m"


class Log:
    @staticmethod
    def section(title: str):
        w = 74
        print(f"\n{C.BOLD}{C.CYAN}{'━'*w}")
        print(f"  {title}")
        print(f"{'━'*w}{C.RESET}")

    @staticmethod
    def step(msg: str):
        print(f"\n{C.BOLD}{C.BLUE}▶ {msg}{C.RESET}")

    @staticmethod
    def ok(msg: str):
        print(f"{C.GREEN}  ✓ {msg}{C.RESET}")

    @staticmethod
    def warn(msg: str):
        print(f"{C.YELLOW}  ⚠  {msg}{C.RESET}")

    @staticmethod
    def err(msg: str):
        print(f"{C.RED}  ✗ {msg}{C.RESET}")

    @staticmethod
    def info(msg: str):
        print(f"{C.DIM}  · {msg}{C.RESET}")

    @staticmethod
    def tool_call(tool: str, args: str, result: str):
        print(f"\n{C.BOLD}{C.MAGENTA}  ╔═ TOOL CALL ══════════════════════════════════╗{C.RESET}")
        print(f"{C.MAGENTA}  ║  🔧 {tool}({args}){C.RESET}")
        preview = result[:120].replace("\n", " ") + ("…" if len(result) > 120 else "")
        print(f"{C.MAGENTA}  ║  → {preview}{C.RESET}")
        print(f"{C.MAGENTA}  ╚═══════════════════════════════════════════════╝{C.RESET}")

    @staticmethod
    def answer_box(query: str, answer: str, meta: Dict[str, Any]):
        w = 74
        print(f"\n{C.BOLD}{C.GREEN}{'═'*w}")
        print(f"  FINAL ANSWER")
        print(f"{'═'*w}{C.RESET}")
        print(f"{C.BOLD}  Q: {query}{C.RESET}\n")
        for line in textwrap.wrap(answer, w - 4):
            print(f"  {line}")
        print(f"\n{C.DIM}{'─'*w}")
        print(f"  Tools used   : {meta.get('tools_used', [])}")
        print(f"  Tool calls   : {meta.get('tool_call_count', 0)}")
        print(f"  Sources      : {meta.get('source_count', 0)}")
        print(f"  Elapsed      : {meta.get('elapsed', 0)}s")
        print(f"{'─'*w}{C.RESET}")



In [33]:


# ─────────────────────────────────────────────────────────────────────
# DEMO CORPUS  (8 rich inline documents across NLP/AI research)
# Referenced from publicly available papers on arXiv
# ─────────────────────────────────────────────────────────────────────
CORPUS: List[Dict[str, str]] = [
    {
        "id": "toolformer",
        "source": "Toolformer: Language Models Can Teach Themselves to Use Tools — Schick et al. (2023)",
        "url": "https://arxiv.org/abs/2302.04761",
        "content": """
Toolformer is a model trained to decide which APIs to call, when to call them, what arguments
to pass, and how to best incorporate the results into future token prediction. Toolformer is
built on GPT-J (6B parameters) and is self-supervised: it generates its own training data
showing where API calls should be inserted.

The key insight is that an API call is useful only if it reduces perplexity on future tokens.
Toolformer filters candidate call positions by computing: L_i - L_i^+ where L_i is the
cross-entropy loss on tokens after position i without the API result, and L_i^+ is the loss
when the API result is prepended to those tokens. Only API calls with positive loss reduction
(threshold τ = 2.0) are kept.

Toolformer uses five tools: Calculator (arithmetic), Calendar (date lookup), Wikipedia Search,
Machine Translation, and Question Answering. API calls are formatted inline as:
  [API_NAME(input) → result]

During training, ~10% of tokens become API call sites. During inference the model generates
[API_NAME(... and the generation pauses, the API is called, the result is inserted, and
generation resumes. This is the core of the Toolformer paradigm.

Toolformer (6B) outperforms GPT-3 (175B) on math tasks (SVAMP: 29.4 vs 14.0) and on
temporal reasoning by leveraging the Calendar tool. It achieves this without increasing
the parameter count, showing that tool use dramatically amplifies effective model capability.
"""
    },
    {
        "id": "react",
        "source": "ReAct: Synergizing Reasoning and Acting in Language Models — Yao et al. (2023)",
        "url": "https://arxiv.org/abs/2210.03629",
        "content": """
ReAct (Reasoning + Acting) interleaves chain-of-thought reasoning traces with task-specific
actions in a unified framework. Unlike standard RAG (retrieve once → generate) or pure
chain-of-thought (no external calls), ReAct agents alternate between:
  Thought: ... (internal reasoning)
  Action: tool_name[input]
  Observation: tool_result
  Thought: ... (updated reasoning)
  → Final Answer

The ReAct pattern allows the model to track state, handle errors from tool calls, and
refine its strategy dynamically. In HotpotQA multi-hop question answering, ReAct achieves
50.3 EM vs. 28.7 for act-only and 29.4 for chain-of-thought only. On ALFWorld (household
task simulation), ReAct achieves 71% success vs. 45% for act-only.

ReAct reduces hallucinations because every factual claim can be grounded in an Observation.
The interleaving of Thought and Action allows error recovery: if an Action fails or returns
unhelpful results, the next Thought can adjust the strategy.

LangChain implements ReAct agents natively via `create_react_agent()` using the hub prompt
`hwchase17/react`. The agent loop runs: format_scratchpad → llm → parse_action →
execute_tool → append_observation → repeat until Final Answer.
"""
    },
    {
        "id": "flare",
        "source": "FLARE: Active Retrieval Augmented Generation — Jiang et al. (2023)",
        "url": "https://arxiv.org/abs/2305.06983",
        "content": """
FLARE (Forward-Looking Active REtrieval) is a method where the language model decides WHEN
to retrieve during long-form generation. Unlike standard RAG (retrieve once at the start),
FLARE monitors the model's confidence on each generated token.

When the model generates a token with probability below a threshold (p < δ, typically 0.4),
it treats that span as a "low-confidence claim" and triggers retrieval. The tentative tokens
become the retrieval query. Retrieved documents are prepended to the context and generation
restarts from the uncertainty point.

FLARE outperforms standard RAG and iterative RAG on long-form generation tasks:
StrategyQA: FLARE 60.1 vs. standard-RAG 57.8
ASQA (answering ambiguous questions): FLARE achieves EM-recall of 43.1 vs. 39.7
2WikiMultihopQA: FLARE 46.2 vs. one-time RAG 37.9

The key algorithmic steps:
1. Generate sentence with current context
2. Check if any token probability < δ
3. If yes: mask low-probability tokens, form retrieval query from remaining tokens
4. Retrieve relevant documents
5. Regenerate the sentence using retrieved context
6. Repeat for each sentence until generation is complete
"""
    },
    {
        "id": "transformer",
        "source": "Attention Is All You Need — Vaswani et al. (2017)",
        "url": "https://arxiv.org/abs/1706.03762",
        "content": """
The Transformer architecture eliminates recurrence entirely, relying solely on self-attention
to compute representations of input and output. The encoder maps an input sequence to
continuous representations; the decoder generates output tokens autoregressively.

Scaled Dot-Product Attention: Attention(Q,K,V) = softmax(Q·K^T / √d_k) · V

Multi-Head Attention: MHA(Q,K,V) = Concat(head_1,...,head_h) · W^O
  where head_i = Attention(Q·W_i^Q, K·W_i^K, V·W_i^V)

The base model uses h=8 heads, d_model=512, d_k=d_v=64. The large model uses h=16, d_model=1024.
Position-wise Feed-Forward: FFN(x) = max(0, xW_1+b_1)W_2+b_2, inner dim d_ff=2048.
Positional encodings: PE(pos,2i) = sin(pos/10000^(2i/d_model)).

Results: WMT 2014 EN→DE: 28.4 BLEU (prior SOTA 26.0). EN→FR: 41.0 BLEU.
Training: Adam optimizer, β1=0.9, β2=0.98, ε=1e-9.
Warmup schedule: lr = d_model^(-0.5) · min(step^(-0.5), step · warmup^(-1.5)), warmup=4000.
Base model: 65M parameters, 12h on 8 P100 GPUs. Large model: 213M parameters, 3.5 days.
"""
    },
    {
        "id": "rag_paper",
        "source": "RAG: Retrieval-Augmented Generation for Knowledge-Intensive NLP — Lewis et al. (2020)",
        "url": "https://arxiv.org/abs/2005.11401",
        "content": """
RAG combines a pre-trained parametric seq2seq model (generator) with a non-parametric dense
vector retrieval component (retriever) into a single trainable system. The retriever encodes
the input query and retrieves top-k documents from a Wikipedia index; the generator produces
the output conditioned on both the input and retrieved documents.

Two formulations:
  RAG-Sequence: p(y|x) = Σ_z p_η(z|x) · p_θ(y|x,z)  [same doc for entire output]
  RAG-Token:    p(y|x) = Π_i Σ_z p_η(z|x) · p_θ(y_i|x,z,y_{1:i-1})  [doc per token]

The retriever uses Dense Passage Retrieval (DPR) with dual BERT encoders. Documents are
indexed as FAISS inner-product index over 21M Wikipedia 100-word passages.

Results on Open-Domain QA:
  Natural Questions: RAG 44.5 EM vs. T5 42.1, DPR 41.5
  WebQuestions: RAG 45.5 vs. T5 37.4
  CuratedTrec: RAG 52.0 vs. BERT 33.0
  TriviaQA: RAG 56.1 vs. T5 60.5

The non-parametric memory (document index) can be updated without retraining the model,
giving RAG an advantage over purely parametric models for time-sensitive knowledge.
"""
    },
    {
        "id": "toolken",
        "source": "ToolkenGPT: Augmenting Frozen Language Models with Massive Tools via Tool Embeddings — Hao et al. (2023)",
        "url": "https://arxiv.org/abs/2305.11554",
        "content": """
ToolkenGPT introduces a fundamentally different approach to tool use: each tool is represented
as a special "toolken" — a learnable embedding vector added to the LM's vocabulary. When the
model generates a toolken, execution switches to a specialized tool-calling mode, where the
model generates the tool arguments, the tool is called, and the result is fed back.

Unlike Toolformer (which fine-tunes the entire model), ToolkenGPT only trains the small set
of toolken embeddings while keeping the base LM frozen. This allows scaling to thousands of
tools without retraining, as each new tool only requires training one new embedding vector.

Toolken training: given a demonstration of a tool call, optimize the toolken embedding to
maximize likelihood of the tool-call context. At inference, the toolken competes with
regular vocabulary tokens in the softmax distribution.

ToolkenGPT results on numeric reasoning:
  MGSM (multilingual math): 84.9% accuracy with tool use vs. 52.7% without
  NumGLUE: 81.3% vs. 19.5% (plain LLaMA-2)
  GSM8K: 90.5% with tools vs. 58.1% plain

The approach works even with LLMs not specifically trained for tool use (LLaMA-2, GPT-J),
showing that tool use capability can be grafted onto existing frozen models efficiently.
"""
    },
    {
        "id": "langchain_agents",
        "source": "LangChain Agents & Tools Documentation",
        "url": "https://python.langchain.com/docs/concepts/agents/",
        "content": """
LangChain agents use a language model as a reasoning engine to decide which actions to take.
The core agent loop: format observation → run LLM → parse action → execute tool →
append observation → repeat until the LLM outputs a final answer (or max iterations reached).

Key agent types in LangChain:
  1. ReAct Agent (create_react_agent) — Thought/Action/Observation loop
  2. OpenAI Function Calling Agent — uses OpenAI function-call API
  3. OpenAI Tools Agent — uses OpenAI parallel tool calling
  4. Structured Chat Agent — multi-input tools, JSON action schema
  5. Self-Ask with Search — automatically asks sub-questions

Tool definition:
  Tool(name="KB", func=retrieval_fn, description="Search knowledge base for a topic")

AgentExecutor wraps the agent and tools, handles the loop, intermediate steps, and errors:
  executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=6,
                           handle_parsing_errors=True, return_intermediate_steps=True)

LCEL (LangChain Expression Language) allows composing chains with the | pipe operator:
  chain = prompt | llm | StrOutputParser()
  rag_chain = {"context": retriever | format_docs, "query": RunnablePassthrough()} | prompt | llm

Memory integration: ConversationBufferMemory, ConversationSummaryMemory for multi-turn sessions.
"""
    },
    {
        "id": "chromadb",
        "source": "ChromaDB: The Open-Source AI Application Database",
        "url": "https://docs.trychroma.com/",
        "content": """
ChromaDB is an open-source vector database designed for AI applications. It supports:
  • In-memory and persistent storage (SQLite + HNSW index via hnswlib)
  • Python and JavaScript clients
  • Metadata filtering combined with vector search
  • Multiple distance metrics: cosine, L2, inner product
  • Multi-modal embeddings (text, images, custom)

Core operations:
  client = chromadb.PersistentClient(path="./chroma_db")
  col    = client.get_or_create_collection("name", metadata={"hnsw:space": "cosine"})
  col.add(documents=["text..."], ids=["id1"], embeddings=[[0.1, 0.2, ...]])
  results = col.query(query_embeddings=[[0.1,...]], n_results=4,
                      where={"source": {"$eq": "paper1"}})

LangChain integration (MMR retrieval):
  vs = Chroma.from_documents(docs, embedding_fn, persist_directory="./chroma")
  retriever = vs.as_retriever(search_type="mmr",
                              search_kwargs={"k": 4, "fetch_k": 8, "lambda_mult": 0.65})

MMR (Maximal Marginal Relevance) balances relevance and diversity:
  MMR = arg max_d [ λ·sim(d,q) - (1-λ)·max_{d'∈S} sim(d,d') ]
where λ=0.65 gives slight preference to relevance over diversity.

HNSW index complexity: O(log n) for both insert and query. ChromaDB automatically
rebuilds the HNSW graph as documents are added, maintaining connectivity.
"""
    },
]



In [34]:

# ─────────────────────────────────────────────────────────────────────
# DOCUMENT PROCESSOR
# ─────────────────────────────────────────────────────────────────────
class DocumentProcessor:
    """Convert raw corpus entries into chunked LangChain Documents."""

    def __init__(self, cfg: Config):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=cfg.CHUNK_SIZE,
            chunk_overlap=cfg.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
            length_function=len,
        )

    def _try_load_pdf(self, url: str, local_path: str) -> List[Document]:
        """Attempt to download and parse a PDF. Returns empty list on failure."""
        try:
            import urllib.request
            if not Path(local_path).exists():
                Log.info(f"Downloading PDF: {url}")
                urllib.request.urlretrieve(url, local_path)
                Log.ok(f"Saved → {local_path}")
            from langchain_community.document_loaders import PyPDFLoader
            loader = PyPDFLoader(local_path)
            pages = loader.load()
            Log.ok(f"Loaded PDF: {len(pages)} pages")
            return pages
        except Exception as exc:
            Log.warn(f"PDF load failed ({exc}), using inline corpus")
            return []

    def build_documents(self) -> List[Document]:
        """Convert corpus dict entries into LangChain Documents."""
        # Try to pull the Toolformer PDF as primary source
        extra: List[Document] = self._try_load_pdf(
            url="https://arxiv.org/pdf/2302.04761",
            local_path="./toolformer.pdf",
        )
        for d in extra:
            d.metadata.update({
                "source": "Toolformer (Schick et al., 2023)",
                "url": "https://arxiv.org/abs/2302.04761",
                "doc_id": "toolformer_pdf",
            })

        inline: List[Document] = []
        for entry in CORPUS:
            doc = Document(
                page_content=entry["content"].strip(),
                metadata={
                    "source": entry["source"],
                    "url": entry["url"],
                    "doc_id": entry["id"],
                },
            )
            inline.append(doc)

        all_docs = extra + inline
        Log.ok(f"Total raw documents: {len(all_docs)}")

        chunks = self.splitter.split_documents(all_docs)

        # Enrich chunk metadata
        seen: set[str] = set()
        unique: List[Document] = []
        for i, c in enumerate(chunks):
            h = hashlib.md5(c.page_content.encode()).hexdigest()[:10]
            if h in seen:
                continue
            seen.add(h)
            c.metadata["chunk_id"]   = i
            c.metadata["chunk_hash"] = h
            c.metadata["char_len"]   = len(c.page_content)
            unique.append(c)

        Log.ok(f"Unique chunks after deduplication: {len(unique)}")
        return unique



In [35]:

# ─────────────────────────────────────────────────────────────────────
# VECTOR STORE BUILDER
# ─────────────────────────────────────────────────────────────────────
class VectorStore:
    """ChromaDB + HuggingFace embeddings wrapper."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.embeddings: Optional[HuggingFaceEmbeddings] = None
        self.store: Optional[Chroma] = None

    def init(self, chunks: List[Document]) -> "VectorStore":
        Log.step("Initialising HuggingFace Embeddings")
        Log.info(f"Model : {self.cfg.EMBEDDING_MODEL}")
        Log.info(f"Device: {self.cfg.EMBEDDING_DEVICE}")

        self.embeddings = HuggingFaceEmbeddings(
            model_name=self.cfg.EMBEDDING_MODEL,
            model_kwargs={"device": self.cfg.EMBEDDING_DEVICE},
            encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
        )
        Log.ok("Embedding model loaded (local, no API key required)")

        Log.step("Building ChromaDB Vector Store")
        client = chromadb.PersistentClient(
            path=self.cfg.CHROMA_PERSIST_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        # Fresh collection every run
        try:
            client.delete_collection(self.cfg.CHROMA_COLLECTION)
        except Exception:
            pass

        self.store = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            client=client,
            collection_name=self.cfg.CHROMA_COLLECTION,
            collection_metadata={"hnsw:space": "cosine"},
        )
        n = self.store._collection.count()
        Log.ok(f"Indexed {n} vectors in ChromaDB → {self.cfg.CHROMA_PERSIST_DIR}")
        return self

    def mmr_retriever(self):
        """Return a diversity-aware MMR retriever."""
        return self.store.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": self.cfg.TOP_K,
                "fetch_k": self.cfg.FETCH_K,
                "lambda_mult": self.cfg.MMR_LAMBDA,
            },
        )

    def similarity_search(self, query: str, k: int = 4) -> List[Tuple[Document, float]]:
        """Direct cosine-similarity search with scores."""
        return self.store.similarity_search_with_relevance_scores(query, k=k)

    def get_context_str(self, query: str, k: int = 4) -> str:
        """Retrieve and format top-k docs as a numbered string."""
        results = self.similarity_search(query, k=k)
        parts = []
        for i, (doc, score) in enumerate(results, 1):
            src = doc.metadata.get("source", "Unknown")
            url = doc.metadata.get("url", "")
            parts.append(
                f"[{i}] {src}\n"
                f"    URL: {url}\n"
                f"    Relevance: {score:.3f}\n"
                f"    {doc.page_content.strip()}"
            )
        return "\n\n".join(parts)


In [36]:


# ─────────────────────────────────────────────────────────────────────
# TOOL REGISTRY
# ─────────────────────────────────────────────────────────────────────
@dataclass
class ToolResult:
    tool_name: str
    args: str
    output: str
    success: bool
    error: Optional[str] = None
    latency_ms: int = 0


class ToolRegistry:
    """
    Registry of all tools available to the Tool-RAG system.
    Tools are pure functions: (args_str, context) → str result.
    """

    def __init__(self, vs: VectorStore, cfg: Config):
        self._vs  = vs
        self._cfg = cfg
        self._registry: Dict[str, Callable[[str], str]] = {}
        self._register_all()

    # ── Registration ─────────────────────────────────────────────────
    def _register_all(self):
        self.register("KB",        self._tool_kb,       "Knowledge base semantic search")
        self.register("CALC",      self._tool_calc,     "Safe arithmetic evaluator")
        self.register("DATE",      self._tool_date,     "Date arithmetic and formatting")
        self.register("COMPARE",   self._tool_compare,  "Side-by-side concept comparison from KB")
        self.register("SUMMARIZE", self._tool_summarize,"Summarise KB results for a topic")
        self.register("FACTCHECK", self._tool_factcheck,"Verify a claim against the KB")
        self.register("DEFINE",    self._tool_define,   "Retrieve definition/explanation of concept")

    def register(self, name: str, fn: Callable[[str], str], description: str):
        self._registry[name] = fn

    def call(self, tool_name: str, args: str) -> ToolResult:
        """Execute a tool by name. Returns structured ToolResult."""
        name = tool_name.upper().strip()
        t0 = time.perf_counter()
        if name not in self._registry:
            return ToolResult(
                tool_name=name, args=args, output="",
                success=False, error=f"Unknown tool '{name}'"
            )
        try:
            output = self._registry[name](args.strip())
            ms = int((time.perf_counter() - t0) * 1000)
            return ToolResult(tool_name=name, args=args, output=output,
                              success=True, latency_ms=ms)
        except Exception as exc:
            ms = int((time.perf_counter() - t0) * 1000)
            return ToolResult(
                tool_name=name, args=args, output="",
                success=False, error=str(exc), latency_ms=ms
            )

    def list_tools(self) -> str:
        """Return a formatted tool list for prompt injection."""
        lines = []
        for name, fn in self._registry.items():
            lines.append(f"  [TOOL:{name}(query)]")
        return "\n".join(lines)

    # ── Tool Implementations ─────────────────────────────────────────

    def _tool_kb(self, query: str) -> str:
        """Semantic search over ChromaDB."""
        return self._vs.get_context_str(query, k=self._cfg.TOP_K)

    def _tool_calc(self, expr: str) -> str:
        """
        Safe arithmetic evaluator supporting: +,-,*,/,**,//,%,sqrt,log,abs,round.
        Uses AST-based evaluation — no exec/eval on raw strings.
        """
        expr = expr.strip()
        # Map friendly names
        expr = re.sub(r'\bsqrt\s*\(', 'math.sqrt(', expr)
        expr = re.sub(r'\blog\s*\(', 'math.log(', expr)
        expr = re.sub(r'\babs\s*\(', 'abs(', expr)
        expr = re.sub(r'\bround\s*\(', 'round(', expr)
        expr = re.sub(r'\bpi\b', str(math.pi), expr)
        expr = re.sub(r'\be\b', str(math.e), expr)

        allowed_nodes = (
            ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
            ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv,
            ast.Mod, ast.Pow, ast.USub, ast.UAdd,
            ast.Call, ast.Attribute, ast.Name, ast.Load,
        )
        allowed_names = {"math": math, "abs": abs, "round": round}

        try:
            tree = ast.parse(expr, mode="eval")
            for node in ast.walk(tree):
                if not isinstance(node, allowed_nodes):
                    raise ValueError(f"Disallowed node: {type(node).__name__}")
            result = eval(  # noqa: S307 — safe because AST is validated
                compile(tree, "<calc>", "eval"),
                {"__builtins__": {}},
                allowed_names,
            )
            return f"{expr} = {result}"
        except Exception as exc:
            return f"CALC ERROR: {exc}"

    def _tool_date(self, query: str) -> str:
        """Parse date expressions and compute results."""
        today = datetime.date.today()
        q = query.lower().strip()
        if q in ("today", "now", "current date"):
            return today.strftime("%A, %B %d, %Y")
        if "days since" in q:
            m = re.search(r"(\d{4}-\d{2}-\d{2})", q)
            if m:
                then = datetime.date.fromisoformat(m.group(1))
                delta = today - then
                return f"{delta.days} days since {m.group(1)}"
        if "days until" in q:
            m = re.search(r"(\d{4}-\d{2}-\d{2})", q)
            if m:
                then = datetime.date.fromisoformat(m.group(1))
                delta = then - today
                return f"{delta.days} days until {m.group(1)}"
        if "year" in q:
            return str(today.year)
        return f"Today is {today.isoformat()}. Query '{query}' not fully understood."

    def _tool_compare(self, query: str) -> str:
        """Compare two concepts by retrieving context for each."""
        # Expect format: "concept_a vs concept_b" or "concept_a, concept_b"
        parts = re.split(r"\s+vs\.?\s+|\s*,\s*", query, maxsplit=1)
        if len(parts) == 2:
            a, b = parts
            ctx_a = self._vs.get_context_str(a.strip(), k=2)
            ctx_b = self._vs.get_context_str(b.strip(), k=2)
            return (
                f"=== {a.strip()} ===\n{ctx_a}\n\n"
                f"=== {b.strip()} ===\n{ctx_b}"
            )
        return self._vs.get_context_str(query, k=4)

    def _tool_summarize(self, topic: str) -> str:
        """Retrieve and concatenate top chunks for summarization."""
        return self._vs.get_context_str(topic, k=self._cfg.TOP_K)

    def _tool_factcheck(self, claim: str) -> str:
        """Retrieve evidence for or against a claim."""
        ctx = self._vs.get_context_str(claim, k=3)
        return f"Evidence retrieved for claim: '{claim}'\n\n{ctx}"

    def _tool_define(self, concept: str) -> str:
        """Retrieve definition/explanation of a concept."""
        return self._vs.get_context_str(concept, k=3)



In [37]:


# ─────────────────────────────────────────────────────────────────────
# TOOL-RAG PROMPTS
# ─────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """\
You are a Tool-RAG assistant that generates answers by calling tools INLINE during your response.
You have access to these tools — insert them literally in your text where facts are needed:

  [TOOL:KB(search query)]        — search knowledge base for information
  [TOOL:CALC(math expression)]   — compute arithmetic (e.g. 512/8, sqrt(768))
  [TOOL:DATE(date query)]        — resolve date questions (today, year, days since YYYY-MM-DD)
  [TOOL:COMPARE(A vs B)]         — retrieve context on two topics for comparison
  [TOOL:SUMMARIZE(topic)]        — retrieve and summarise a topic
  [TOOL:FACTCHECK(claim)]        — retrieve evidence to verify a claim
  [TOOL:DEFINE(concept)]         — retrieve a definition/explanation

RULES:
1. Insert tool calls inline EXACTLY where the factual information should appear, like:
   "The Transformer uses [TOOL:KB(scaled dot product attention)] as its core operation."
2. Do NOT call the same tool with the same arguments twice.
3. You may call up to {max_tool_calls} tools in one response.
4. After all [TOOL:...] markers are resolved by the engine, your answer will be complete.
5. If a question is purely about dates or arithmetic, use CALC or DATE instead of KB.
6. Ground every factual claim in a tool call — never fabricate facts.
7. Always cite sources as [Source: document title] after each factual statement.
"""

INLINE_GEN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", (
        "Conversation history:\n{history}\n\n"
        "User question: {question}\n\n"
        "Write your answer now, inserting [TOOL:NAME(args)] markers where facts are needed:"
    )),
])

FINAL_SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a precise, scholarly assistant. Below is a draft answer where [TOOL:NAME(args)] "
        "markers have been replaced with actual retrieved results inside <<RESULT>>...</RESULT>> blocks.\n\n"
        "Your task:\n"
        "1. Rewrite the answer fluently, integrating all tool results naturally.\n"
        "2. Preserve all factual content from the tool results — do not omit or alter facts.\n"
        "3. Add [Source: N] citations referencing numbered sources.\n"
        "4. The final answer must be self-contained, well-structured, and directly answer the question.\n"
        "5. Do NOT include any [TOOL:...] markers or <<RESULT>> tags in your final output."
    )),
    ("human", (
        "Original question: {question}\n\n"
        "Draft with tool results:\n{draft_with_results}\n\n"
        "Tool results summary:\n{tool_summary}\n\n"
        "Write the final polished answer:"
    )),
])

SELF_CRITIQUE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a critical quality evaluator for AI-generated answers.\n"
        "Score the answer on: factual_accuracy (0-10), completeness (0-10), "
        "tool_usage_quality (0-10), citation_quality (0-10).\n"
        "Also list any issues and provide an improved_answer only if overall_score < 7.\n"
        "Respond ONLY with valid JSON. No markdown fences, no preamble."
    )),
    ("human", (
        "Question: {question}\n"
        "Answer: {answer}\n"
        "Tools used: {tools_used}\n\n"
        "Evaluate and respond with JSON keys: "
        "factual_accuracy, completeness, tool_usage_quality, citation_quality, "
        "overall_score, issues (list), improved_answer (str or null)"
    )),
])



In [38]:

# ─────────────────────────────────────────────────────────────────────
# TOOL CALL PARSER
# ─────────────────────────────────────────────────────────────────────
# Matches:  [TOOL:TOOLNAME(args)]  or  [TOOL: TOOLNAME ( args )]
TOOL_PATTERN = re.compile(
    r"\[TOOL\s*:\s*([A-Z_]+)\s*\(([^)]*)\)\]",
    re.IGNORECASE,
)


def parse_tool_calls(text: str) -> List[Tuple[str, str, str]]:
    """
    Find all tool call markers in `text`.
    Returns list of (full_match, tool_name, args).
    """
    results = []
    for m in TOOL_PATTERN.finditer(text):
        results.append((m.group(0), m.group(1).upper(), m.group(2)))
    return results



In [39]:

# ─────────────────────────────────────────────────────────────────────
# CORE TOOL-RAG ENGINE
# ─────────────────────────────────────────────────────────────────────
class ToolRAGEngine:
    """
    Toolformer-style RAG engine.

    Pipeline:
      1. LLM generates a draft with inline [TOOL:NAME(args)] markers.
      2. Engine parses & executes each tool call (knowledge base, calculator, etc.)
      3. Results are substituted back into the draft.
      4. LLM synthesizes a final, polished, cited answer.
      5. Self-critique evaluates & optionally improves the answer.
    """

    def __init__(self, cfg: Config, vs: VectorStore):
        self.cfg = cfg
        self.vs  = vs
        self.llm: Optional[ChatOllama] = None
        self.tools: Optional[ToolRegistry]  = None
        self._conversation_history: List[Dict[str, str]] = []
        self._output_parser = StrOutputParser()

    def init_llm(self) -> "ToolRAGEngine":
        Log.step("Connecting to Azure OpenAI")
        self.llm = ChatOllama(
            base_url=getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434'),
            model=getattr(config, 'OLLAMA_MODEL', 'qwen2.5-coder:7b'),
            temperature=getattr(config, 'LLM_TEMPERATURE', 0.0),
            num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
        )
        Log.ok(f"LLM ready — deployment: {self.cfg.OLLAMA_MODEL}")
        return self

    def init_tools(self) -> "ToolRAGEngine":
        self.tools = ToolRegistry(self.vs, self.cfg)
        tool_names = list(self.tools._registry.keys())
        Log.ok(f"Tools registered: {tool_names}")
        return self

    # ── Step 1: Generate draft with inline tool markers ───────────────
    def _generate_draft(self, question: str) -> str:
        Log.step("Step 1 · LLM generating draft with inline tool markers")
        history_str = "\n".join(
            f"{h['role'].upper()}: {h['content']}"
            for h in self._conversation_history[-6:]  # last 3 turns
        ) if self._conversation_history else "None"

        chain = INLINE_GEN_PROMPT | self.llm | self._output_parser
        draft = chain.invoke({
            "max_tool_calls": self.cfg.MAX_TOOL_ITERATIONS,
            "question": question,
            "history": history_str,
        })
        Log.info(f"Draft length: {len(draft)} chars")
        return draft

    # ── Step 2: Execute all tool calls in the draft ───────────────────
    def _execute_tools(self, draft: str) -> Tuple[str, List[ToolResult]]:
        """
        Parse draft for [TOOL:NAME(args)] markers, execute each,
        substitute result into the draft, return annotated draft + all results.
        """
        Log.step("Step 2 · Executing inline tool calls")
        all_results: List[ToolResult] = []
        annotated   = draft
        seen_calls: set[str] = set()
        iterations  = 0

        while True:
            calls = parse_tool_calls(annotated)
            if not calls or iterations >= self.cfg.MAX_TOOL_ITERATIONS:
                break

            found_new = False
            for full_match, tool_name, args in calls:
                call_key = f"{tool_name}::{args}"
                if call_key in seen_calls:
                    # Replace duplicate with a note and skip
                    annotated = annotated.replace(
                        full_match,
                        f"[duplicate call to {tool_name} omitted]",
                        1,
                    )
                    continue

                seen_calls.add(call_key)
                found_new = True
                result = self.tools.call(tool_name, args)
                all_results.append(result)
                Log.tool_call(tool_name, args,
                              result.output if result.success else f"ERROR: {result.error}")

                if result.success:
                    replacement = f"<<RESULT>>{result.output}</RESULT>>"
                else:
                    replacement = f"[Tool {tool_name} failed: {result.error}]"

                annotated = annotated.replace(full_match, replacement, 1)
                iterations += 1

            if not found_new:
                break

        Log.ok(f"Executed {len(all_results)} tool call(s) in {iterations} iteration(s)")
        return annotated, all_results

    # ── Step 3: Synthesize final answer ───────────────────────────────
    def _synthesize(
        self,
        question: str,
        draft_with_results: str,
        tool_results: List[ToolResult],
    ) -> str:
        Log.step("Step 3 · LLM synthesizing final answer")
        tool_summary = "\n".join(
            f"• {r.tool_name}({r.args}) → "
            + (r.output[:200].replace("\n", " ") if r.success else f"ERROR: {r.error}")
            for r in tool_results
        ) or "No tools were called."

        chain = FINAL_SYNTHESIS_PROMPT | self.llm | self._output_parser
        answer = chain.invoke({
            "question": question,
            "draft_with_results": draft_with_results,
            "tool_summary": tool_summary,
        })
        return answer

    # ── Step 4: Self-critique ─────────────────────────────────────────
    def _self_critique(
        self,
        question: str,
        answer: str,
        tool_results: List[ToolResult],
    ) -> Tuple[str, Dict[str, Any]]:
        Log.step("Step 4 · Self-critique evaluation")
        tools_used = [f"{r.tool_name}({r.args[:40]})" for r in tool_results]

        chain = SELF_CRITIQUE_PROMPT | self.llm | self._output_parser
        raw = chain.invoke({
            "question": question,
            "answer": answer,
            "tools_used": str(tools_used),
        })

        try:
            clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            critique = json.loads(clean)
        except json.JSONDecodeError:
            Log.warn("Critique JSON parse failed — skipping improvement")
            critique = {"overall_score": 8, "issues": [], "improved_answer": None}

        score = critique.get("overall_score", 8)
        Log.ok(f"Quality score: {score}/10  |  Issues: {critique.get('issues', [])}")

        final = critique.get("improved_answer") if score < 7 else None
        if final:
            Log.info("Using critic-improved answer")
            return final, critique
        return answer, critique

    # ── Main entry point ──────────────────────────────────────────────
    def query(
        self,
        question: str,
        run_critique: bool = True,
    ) -> Dict[str, Any]:
        """
        Full Tool-RAG pipeline:
          Draft → Tool Execution → Synthesis → Critique → Final Answer
        """
        t0 = time.time()
        Log.section(f"QUERY: {question}")

        # 1. LLM drafts with inline tool markers
        draft = self._generate_draft(question)

        # 2. Execute every tool marker found in draft
        annotated_draft, tool_results = self._execute_tools(draft)

        # 3. LLM synthesizes final answer
        answer = self._synthesize(question, annotated_draft, tool_results)

        # 4. Optional self-critique
        critique: Dict[str, Any] = {}
        if run_critique and self.llm is not None:
            answer, critique = self._self_critique(question, answer, tool_results)

        # 5. Collect unique sources
        sources = list({
            doc.metadata.get("source", "")
            for r in tool_results if r.success
            for doc in []  # sources come from tool output text
        })
        # Extract source lines from tool outputs
        source_set: set[str] = set()
        for r in tool_results:
            for line in r.output.splitlines():
                line = line.strip()
                if line.startswith("[") and "] " in line:
                    # Format: [1] Source name
                    src_part = line.split("] ", 1)[1].strip()
                    if src_part and not src_part.startswith("URL"):
                        source_set.add(src_part)
        sources = list(source_set)

        elapsed = round(time.time() - t0, 2)
        tools_used_names = list({r.tool_name for r in tool_results})

        result: Dict[str, Any] = {
            "query":           question,
            "answer":          answer,
            "draft":           draft,
            "annotated_draft": annotated_draft,
            "tool_results":    tool_results,
            "tools_used":      tools_used_names,
            "tool_call_count": len(tool_results),
            "sources":         sources,
            "source_count":    len(sources),
            "critique":        critique,
            "critique_score":  critique.get("overall_score"),
            "elapsed":         elapsed,
        }

        # Update conversation history
        self._conversation_history.append({"role": "user",      "content": question})
        self._conversation_history.append({"role": "assistant",  "content": answer})

        Log.answer_box(question, answer, result)
        return result

    def reset_history(self):
        """Clear multi-turn conversation history."""
        self._conversation_history.clear()
        Log.info("Conversation history reset")


In [40]:


# ─────────────────────────────────────────────────────────────────────
# REACT PROMPT
# Must contain: {tools}, {tool_names}, {input}, {agent_scratchpad}
# Optionally:   {chat_history}
# ─────────────────────────────────────────────────────────────────────
REACT_PROMPT_TEMPLATE = """\
You are a Tool-RAG (Toolformer-style) research assistant.
You have access to a knowledge base of AI/NLP research papers and utility tools.

Available tools:
{tools}

Use this format EXACTLY:

Question: the input question you must answer
Thought: reason about what to do
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (Thought/Action/Action Input/Observation can repeat up to 6 times)
Thought: I now know the final answer
Final Answer: the complete, cited answer to the original question

Rules:
- Every factual statement must be grounded in an Observation from a tool.
- Cite sources as [Source: document title] after key claims.
- For math, always use the Calculator tool — never compute in your head.
- For comparisons, use the Comparator tool for structured retrieval.
- If one search doesn't return good results, try different keywords.
- After 5 tool calls, synthesize with what you have.

Begin!

{chat_history}
Question: {input}
Thought:{agent_scratchpad}"""



In [41]:

# ─────────────────────────────────────────────────────────────────────
# TOOL BUILDER
# ─────────────────────────────────────────────────────────────────────
def _make_tools(engine: ToolRAGEngine) -> List[Tool]:
    """Convert ToolRegistry functions into LangChain Tool objects."""
    registry = engine.tools

    def _wrap(tool_name: str) -> callable:
        def _fn(query: str) -> str:
            result = registry.call(tool_name, query)
            if result.success:
                return result.output or "(empty result)"
            return f"Error: {result.error}"
        return _fn

    tools = [
        Tool(
            name="KnowledgeBase",
            func=_wrap("KB"),
            description=(
                "Search the knowledge base (ChromaDB) for information about "
                "Toolformer, ReAct, FLARE, Transformers, RAG, ToolkenGPT, "
                "LangChain, ChromaDB, or any AI/NLP research topic. "
                "Input: a natural language search query (e.g. 'multi-head attention formula')."
            ),
        ),
        Tool(
            name="Calculator",
            func=_wrap("CALC"),
            description=(
                "Evaluate arithmetic expressions safely. Supports: +, -, *, /, **, //, %, "
                "sqrt(), log(), abs(), round(), pi, e. "
                "Input: a math expression string (e.g. '512 / 8', 'sqrt(768)', '28.4 - 26.0')."
            ),
        ),
        Tool(
            name="DateResolver",
            func=_wrap("DATE"),
            description=(
                "Resolve date-related questions. "
                "Input examples: 'today', 'year', 'days since 2017-06-01', 'days until 2025-01-01'."
            ),
        ),
        Tool(
            name="Comparator",
            func=_wrap("COMPARE"),
            description=(
                "Retrieve context for two concepts side-by-side for comparison. "
                "Input format: 'concept_A vs concept_B' "
                "(e.g. 'Toolformer vs ToolkenGPT', 'RAG-Sequence vs RAG-Token')."
            ),
        ),
        Tool(
            name="Summarizer",
            func=_wrap("SUMMARIZE"),
            description=(
                "Retrieve and summarise multiple relevant passages on a topic. "
                "Use when you need a broad overview rather than a specific fact. "
                "Input: a topic string (e.g. 'FLARE active retrieval method')."
            ),
        ),
        Tool(
            name="FactChecker",
            func=_wrap("FACTCHECK"),
            description=(
                "Retrieve evidence supporting or contradicting a specific factual claim. "
                "Input: a factual claim to verify "
                "(e.g. 'Toolformer outperforms GPT-3 on math tasks')."
            ),
        ),
        Tool(
            name="Definer",
            func=_wrap("DEFINE"),
            description=(
                "Retrieve a detailed definition or explanation of a concept. "
                "Input: a concept name "
                "(e.g. 'Maximal Marginal Relevance', 'scaled dot-product attention')."
            ),
        ),
    ]
    return tools



In [42]:

# ─────────────────────────────────────────────────────────────────────
# AGENT BUILDER
# ─────────────────────────────────────────────────────────────────────
def build_langchain_agent(
    engine: ToolRAGEngine,
    verbose: bool = True,
    max_iterations: int = 6,
    use_memory: bool = True,
) -> AgentExecutor:
    """
    Build a LangChain ReAct AgentExecutor backed by the Tool-RAG engine.

    Args:
        engine       : Initialised ToolRAGEngine instance
        verbose      : Print agent reasoning chain
        max_iterations: Hard limit on tool calls per query
        use_memory   : Include sliding-window conversation memory

    Returns:
        AgentExecutor ready to call with executor.invoke({"input": "..."})
    """
    llm   = engine.llm
    tools = _make_tools(engine)

    prompt = PromptTemplate.from_template(REACT_PROMPT_TEMPLATE)

    agent = create_react_agent(
        llm=llm,
        tools=tools,
        prompt=prompt,
    )

    memory = None
    if use_memory:
        memory =  InMemorySaver()

    executor = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=verbose,
        max_iterations=max_iterations,
        handle_parsing_errors=True,
        return_intermediate_steps=True,
        memory=memory,
        early_stopping_method="generate",
    )

    Log.ok(f"LangChain ReAct AgentExecutor built with {len(tools)} tools")
    return executor



In [43]:


# ─────────────────────────────────────────────────────────────────────
# STANDALONE RUN HELPER
# ─────────────────────────────────────────────────────────────────────
def run_agent_query(
    executor: AgentExecutor,
    question: str,
    show_steps: bool = True,
) -> Dict[str, Any]:
    """
    Run a single query through the ReAct agent and return structured output.

    Returns dict with: output, intermediate_steps, tool_call_count
    """
    Log.section(f"REACT AGENT QUERY: {question}")
    t0 = __import__("time").time()

    try:
        result = executor.invoke({"input": question})
    except Exception as exc:
        Log.err(f"Agent error: {exc}")
        return {
            "output": f"Agent encountered an error: {exc}",
            "intermediate_steps": [],
            "tool_call_count": 0,
            "elapsed": round(__import__("time").time() - t0, 2),
        }

    steps = result.get("intermediate_steps", [])
    elapsed = round(__import__("time").time() - t0, 2)

    if show_steps:
        print(f"\n{C.DIM}{'─'*70}")
        print(f"  Agent intermediate steps: {len(steps)}")
        for i, (action, observation) in enumerate(steps, 1):
            print(f"\n  Step {i}: {C.MAGENTA}{action.tool}{C.RESET}({action.tool_input[:60]})")
            obs_preview = str(observation)[:150].replace("\n", " ")
            print(f"    Obs: {C.DIM}{obs_preview}{C.RESET}")
        print(f"{'─'*70}{C.RESET}")

    final = {
        "output":             result.get("output", ""),
        "intermediate_steps": steps,
        "tool_call_count":    len(steps),
        "elapsed":            elapsed,
    }

    w = 70
    print(f"\n{C.BOLD}{C.GREEN}{'═'*w}")
    print(f"  REACT AGENT ANSWER")
    print(f"{'═'*w}{C.RESET}")
    import textwrap
    for line in textwrap.wrap(final["output"], w - 4):
        print(f"  {line}")
    print(f"\n{C.DIM}  Tool calls: {final['tool_call_count']}  |  Elapsed: {elapsed}s{C.RESET}")
    print(f"{C.DIM}{'─'*w}{C.RESET}")

    return final


In [44]:

# ─────────────────────────────────────────────────────────────────────
# DEMO QUERIES  — 8 diverse queries exercising all 7 tools
# ─────────────────────────────────────────────────────────────────────
DEMO_QUERIES = [
    # 1. DEFINE + CALC — factual + arithmetic
    (
        "What is Toolformer's core mechanism for deciding when to call an API, "
        "and what is the perplexity threshold τ multiplied by 2?",
        "Exercises: KB + CALC"
    ),
    # 2. COMPARE — side-by-side comparison
    (
        "Compare Toolformer and ToolkenGPT: how do they differ in how tools are integrated into the LM?",
        "Exercises: COMPARE"
    ),
    # 3. KB + CALC — multi-fact + arithmetic
    (
        "In the original Transformer, how many parameters does each attention head operate on "
        "if d_model=512 and there are 8 heads? Compute d_k.",
        "Exercises: KB + CALC"
    ),
    # 4. FACTCHECK — verification
    (
        "Is it true that Toolformer (6B parameters) outperforms GPT-3 (175B parameters) on the SVAMP math benchmark?",
        "Exercises: FACTCHECK"
    ),
    # 5. FLARE / KB — explanatory
    (
        "Explain how FLARE (Forward-Looking Active REtrieval) decides when to trigger retrieval "
        "during long-form text generation.",
        "Exercises: KB + SUMMARIZE"
    ),
    # 6. DATE + KB — temporal + knowledge
    (
        "The Transformer paper was published in June 2017. How many days ago was that from today? "
        "Also, what BLEU score did it achieve on English-to-German translation?",
        "Exercises: DATE + CALC + KB"
    ),
    # 7. DEFINE — technical concept
    (
        "Define Maximal Marginal Relevance (MMR) as used in ChromaDB retrieval and explain its formula.",
        "Exercises: DEFINE + KB"
    ),
    # 8. Multi-hop — chained reasoning
    (
        "How does ReAct's Thought-Action-Observation loop differ from FLARE's retrieval strategy, "
        "and which achieves better results on multi-hop QA benchmarks?",
        "Exercises: COMPARE + KB + FACTCHECK"
    ),
]


In [45]:

# ─────────────────────────────────────────────────────────────────────
# SYSTEM BUILDER
# ─────────────────────────────────────────────────────────────────────
def build_system(cfg: Config) -> ToolRAGEngine:
    """Full initialization: documents → vectors → LLM → tools."""
    Log.section("TOOL-RAG SYSTEM INITIALIZATION")

    # 1. Documents
    Log.step("Building document corpus")
    processor = DocumentProcessor(cfg)
    chunks    = processor.build_documents()

    # 2. Vector store
    vs = VectorStore(cfg)
    vs.init(chunks)

    # 3. Engine
    engine = ToolRAGEngine(cfg, vs)

    if cfg.is_azure_configured():
        engine.init_llm()
    else:
        Log.warn("Azure OpenAI credentials not configured.")
        Log.warn("Set OLLAMA_BASE_URL and OLLAMA_BASE_URL env vars.")
        Log.warn("Vector store is ready — LLM calls will fail gracefully.")

    engine.init_tools()

    Log.section("SYSTEM READY")
    return engine


In [46]:
# ─────────────────────────────────────────────────────────────────────
# DEMO RUN
# ─────────────────────────────────────────────────────────────────────
def run_demo(engine: ToolRAGEngine, run_critique: bool = True):
    """Run all 8 demo queries sequentially."""
    print(f"\n{C.BOLD}{C.CYAN}Running {len(DEMO_QUERIES)} demo queries...{C.RESET}")

    for i, (query, note) in enumerate(DEMO_QUERIES, 1):
        print(f"\n{C.BOLD}{C.YELLOW}Demo {i}/{len(DEMO_QUERIES)}  [{note}]{C.RESET}")
        try:
            engine.query(query, run_critique=run_critique)
        except Exception as exc:
            Log.err(f"Query failed: {exc}")
        time.sleep(0.5)



In [47]:


# ─────────────────────────────────────────────────────────────────────
# INTERACTIVE CLI  — Tool-RAG engine
# ─────────────────────────────────────────────────────────────────────
def interactive_cli(engine: ToolRAGEngine, run_critique: bool = True):
    print(f"\n{C.BOLD}{C.CYAN}")
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║       Tool-RAG / Toolformer-RAG  ·  Interactive CLI         ║")
    print("║  Commands: 'demo N' | 'reset' | 'tools' | 'quit'            ║")
    print("╚══════════════════════════════════════════════════════════════╝")
    print(C.RESET)
    print(f"{C.DIM}Available tools: KB, CALC, DATE, COMPARE, SUMMARIZE, FACTCHECK, DEFINE{C.RESET}\n")

    while True:
        try:
            user = input(f"{C.BOLD}You:{C.RESET} ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user:
            continue

        cmd = user.lower()
        if cmd in ("quit", "exit", "q"):
            print("Goodbye!")
            break
        if cmd == "reset":
            engine.reset_history()
            continue
        if cmd == "tools":
            print(f"\n{C.MAGENTA}Available tools:{C.RESET}")
            for name in engine.tools._registry:
                print(f"  • {name}")
            continue
        if cmd.startswith("demo"):
            parts = cmd.split()
            n = int(parts[1]) - 1 if len(parts) > 1 and parts[1].isdigit() else 0
            if 0 <= n < len(DEMO_QUERIES):
                query, note = DEMO_QUERIES[n]
                print(f"{C.DIM}Running: {query}{C.RESET}")
                engine.query(query, run_critique=run_critique)
            else:
                print(f"Usage: demo 1-{len(DEMO_QUERIES)}")
            continue

        try:
            engine.query(user, run_critique=run_critique)
        except Exception as exc:
            Log.err(f"Error: {exc}")



In [48]:

# ─────────────────────────────────────────────────────────────────────
# INTERACTIVE CLI  — LangChain ReAct Agent
# ─────────────────────────────────────────────────────────────────────
def interactive_agent_cli(engine: ToolRAGEngine):
    executor = build_langchain_agent(engine, verbose=True)

    print(f"\n{C.BOLD}{C.CYAN}")
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║       Tool-RAG  ·  LangChain ReAct Agent  CLI               ║")
    print("║  Commands: 'quit' to exit                                    ║")
    print("╚══════════════════════════════════════════════════════════════╝")
    print(C.RESET)

    while True:
        try:
            user = input(f"{C.BOLD}You:{C.RESET} ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user or user.lower() in ("quit", "exit", "q"):
            break

        try:
            run_agent_query(executor, user)
        except Exception as exc:
            Log.err(f"Agent error: {exc}")


# ─────────